## 🥈 Geolocation Data Preprocessing
* **Data types & padding:** Converted columns to string formats and used `.str.zfill(5)` on zip codes to keep leading zeros.
* **Text fixing:** Used `"".join()` and `set()` to find Portuguese typos and replaced encoding errors manually.
* **Duplicates:** Removed duplicate GPS entries per zip code by simply using `keep='first'` to keep the dataset small.

## 🥈 Product Category Translation Preprocessing
* **Data types:** Changed the data types from generic object to proper string formats.
* **Text formatting:** Stripped trailing whitespaces, converted all text to lowercase, and removed duplicate categories.
* **Surrogate key:** Reset the index to create a clean sequential ID and checked the maximum character lengths.

## 🥈 Order Items Preprocessing
* **Formatting & Types:** Cast ID columns to strings, rounded prices and freight costs to 2 decimal places, and converted shipping dates into datetime64 format.
* **Data validation:** Checked the data for null values and verified that all ID keys have a standard length of 32 characters.
* **Surrogate key:** Cleaned up the index to build a primary key and reordered columns for better readability.



In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#importing libraries


## Inspecting the data

Checking for null-values and the data types



In [3]:
df1=pd.read_csv(r"C:\Users\User\Desktop\DSI\Abschlussprojekt\Quelldaten\bronze_olist_geolocation_dataset.csv")
df_geolocation= df1.copy()
df_geolocation.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB


## Find missing values

 The value_counts matches the rangeindex from above in both cases. There are no missing values. The columns will be converted into strings.

In [4]:
df_geolocation['geolocation_city'].apply(type).value_counts()
df_geolocation['geolocation_state'].apply(type).value_counts()

geolocation_state
<class 'str'>    1000163
Name: count, dtype: int64

In [5]:
spalten_liste= ['geolocation_city','geolocation_state']
df_geolocation[spalten_liste]=df_geolocation[spalten_liste].astype("string")
df_geolocation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  string 
 4   geolocation_state            1000163 non-null  string 
dtypes: float64(2), int64(1), string(2)
memory usage: 38.2 MB


## Altering of coordinates column

The columns of longitute and latitute will be altered. It is to much information in the column so do only keep one set of coordinates for every zip code.

In [6]:
df_geolocation = df_geolocation.drop_duplicates(subset=['geolocation_zip_code_prefix'], keep='first')
df_geolocation

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
5,1012,-23.547762,-46.635361,são paulo,SP
...,...,...,...,...,...
999774,99955,-28.107588,-52.144019,vila langaro,RS
999780,99970,-28.345143,-51.876926,ciriaco,RS
999786,99910,-27.863500,-52.084760,floriano peixoto,RS
999803,99920,-27.858716,-52.300403,erebango,RS


## Data Cleaning

It starts with checking the state. Is has to be two charakters long in brazil. After that it will checked if there are special characters in the column. The will be converted into snake case and whitespaces will be removed. 


In [7]:
state_lengths = df_geolocation['geolocation_state'].str.len()
state_lengths_count = state_lengths.value_counts()
print(f"Distribution of lengths: {state_lengths_count}")



broken_states = df_geolocation[state_lengths != 2]
if not broken_states.empty:
    print(f" The following lines are incorrect: {broken_states}")
else:
    print(f"There are no data errors!")


Distribution of lengths: geolocation_state
2    19015
Name: count, dtype: Int64
There are no data errors!


In [8]:
df_geolocation['geolocation_state'] = df_geolocation['geolocation_state'].str.strip().str.lower()
string_elements="".join(df_geolocation['geolocation_state'])
unique_strings=sorted(list(set(string_elements)))
print(f" The following characters are in the column {unique_strings}")


 The following characters are in the column ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'i', 'j', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't']


## Data Cleaning

 The portuguese letters and special characters will be removed, as well as the white spaces.
 The dictonary corrects the most common mistakes. 

In [9]:
string_elements="".join(df_geolocation['geolocation_city'])
unique_strings=sorted(list(set(string_elements)))
print(unique_strings)

replacements={'â': 'a', 'í': 'i', 'ó': 'o', 'ô': 'o', 'õ': 'o', 'ü': 'u', '´': '', '`': '', 'º': '' }
df_geolocation['geolocation_city'] = df_geolocation['geolocation_city'].str.lower().str.strip().replace(replacements, regex=True)
df_geolocation['geolocation_city'] = df_geolocation['geolocation_city'].str.replace(r'[^a-z\s\-]', '', regex=True)
df_geolocation['geolocation_city'] = df_geolocation['geolocation_city'].str.strip() 


city_names = df_geolocation[df_geolocation['geolocation_city'].str.contains(' ', na=False)]
clean_city_names=city_names['geolocation_city'].value_counts()
print(clean_city_names)


[' ', "'", ',', '-', '1', '3', '4', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'á', 'â', 'ã', 'ç', 'é', 'ê', 'í', 'ó', 'ô', 'õ', 'ú', 'ü']
geolocation_city
sao paulo              2652
so paulo                528
rio de janeiro          400
belo horizonte          204
porto alegre            142
                       ... 
nova itapirema            1
pereira barreto           1
ilha solteira             1
general salgado           1
bela vista do toldo       1
Name: count, Length: 2866, dtype: Int64


A last check on whitespaces and unique strings as well as correcting spelling mistakes in the city column.

In [10]:
orthography={'so paulo': 'sao paulo', 'jos bonifcio': 'jose bonifacio','jose bonifactio':'jose bonifacio' ,'sopaulo':'sao paulo','sp':'sao paulo','poa':'porto alegre', 'po':'port alegre','mogidascruzes':'mogi das cruzes','biritiba-mirim':'biritiba mirim','santo andr':'santo andre'}
df_geolocation['geolocation_city']=df_geolocation['geolocation_city'].replace(orthography,regex=True) 


print(city_names['geolocation_city'])

whitespaces = df_geolocation[df_geolocation['geolocation_city'].str.startswith(' ') | df_geolocation['geolocation_city'].str.endswith(' ')]

print(whitespaces)


string_elements="".join(df_geolocation['geolocation_city'])
unique_strings=sorted(list(set(string_elements)))
print(unique_strings)

0                    sao paulo
1                    sao paulo
3                    sao paulo
4                    sao paulo
5                     so paulo
                  ...         
999754              agua santa
999772         ipiranga do sul
999774            vila langaro
999786        floriano peixoto
999846    santa cecilia do sul
Name: geolocation_city, Length: 9777, dtype: string
Empty DataFrame
Columns: [geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state]
Index: []
[' ', '-', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


## Data Validating

 Unusal and usual combinations of citys and states will be detected and corrected.

In [11]:
combinations= df_geolocation.groupby(['geolocation_state','geolocation_city'])
top_combinations=combinations.head(20)
unusual_combinations=combinations.tail(20)
print(f"The most common combinations are {top_combinations}")
print(f"Unusual combinations are {unusual_combinations}")

The most common combinations are         geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                              1037       -23.545621       -46.639292   
1                              1046       -23.546081       -46.644820   
3                              1041       -23.544392       -46.639499   
4                              1035       -23.541578       -46.641607   
5                              1012       -23.547762       -46.635361   
...                             ...              ...              ...   
999774                        99955       -28.107588       -52.144019   
999780                        99970       -28.345143       -51.876926   
999786                        99910       -27.863500       -52.084760   
999803                        99920       -27.858716       -52.300403   
999846                        99952       -28.160371       -51.936922   

            geolocation_city geolocation_state  
0                  sao paulo             

## Correcting incorrect distributions and rechecking them

In [12]:
grouped_combinations = df_geolocation.groupby(['geolocation_city', 'geolocation_state']).size().reset_index(name='count')
sorted_combinations = grouped_combinations.sort_values(['geolocation_city', 'count'], ascending=[True, False])


correct_pairs = sorted_combinations.drop_duplicates(subset=['geolocation_city'])

correct_combinations = dict(zip(correct_pairs['geolocation_city'], correct_pairs['geolocation_state']))


df_geolocation['geolocation_state'] = df_geolocation['geolocation_city'].map(correct_combinations)

df_geolocation.info()
df_geolocation['geolocation_state']=df_geolocation['geolocation_state'].astype('string')
df_geolocation.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19015 entries, 0 to 999846
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   geolocation_zip_code_prefix  19015 non-null  int64  
 1   geolocation_lat              19015 non-null  float64
 2   geolocation_lng              19015 non-null  float64
 3   geolocation_city             19015 non-null  string 
 4   geolocation_state            19015 non-null  object 
dtypes: float64(2), int64(1), object(1), string(1)
memory usage: 891.3+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 19015 entries, 0 to 999846
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   geolocation_zip_code_prefix  19015 non-null  int64  
 1   geolocation_lat              19015 non-null  float64
 2   geolocation_lng              19015 non-null  float64
 3   geolocation_city

In [13]:
combinations= df_geolocation.groupby(['geolocation_state','geolocation_city'])
top_combinations=combinations.head(20)
unusual_combinations=combinations.tail(20)
print(f"The most common combinations are {top_combinations}")
print(f"Unusual combinations are {unusual_combinations}")

The most common combinations are         geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                              1037       -23.545621       -46.639292   
1                              1046       -23.546081       -46.644820   
3                              1041       -23.544392       -46.639499   
4                              1035       -23.541578       -46.641607   
5                              1012       -23.547762       -46.635361   
...                             ...              ...              ...   
999774                        99955       -28.107588       -52.144019   
999780                        99970       -28.345143       -51.876926   
999786                        99910       -27.863500       -52.084760   
999803                        99920       -27.858716       -52.300403   
999846                        99952       -28.160371       -51.936922   

            geolocation_city geolocation_state  
0                  sao paulo             

## Removing Duplicates

In [14]:
df_geolocation = df_geolocation.drop_duplicates() # removing duplicates
df_geolocation

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,sp
1,1046,-23.546081,-46.644820,sao paulo,sp
3,1041,-23.544392,-46.639499,sao paulo,sp
4,1035,-23.541578,-46.641607,sao paulo,sp
5,1012,-23.547762,-46.635361,sao paulo,sp
...,...,...,...,...,...
999774,99955,-28.107588,-52.144019,vila langaro,rs
999780,99970,-28.345143,-51.876926,ciriaco,rs
999786,99910,-27.863500,-52.084760,floriano peixoto,rs
999803,99920,-27.858716,-52.300403,erebango,rs


## Converting the zipcode and bring it into 5 figure format

 Converting zipcode into a string format. It is easier to work with, since a zipcode won't be used in metric operations.

In [15]:
df_geolocation["geolocation_zip_code_prefix"]=df_geolocation["geolocation_zip_code_prefix"].astype(str).str.zfill(5)

df_geolocation["geolocation_zip_code_prefix"]=df_geolocation["geolocation_zip_code_prefix"].astype("string")
zip_length= df_geolocation["geolocation_zip_code_prefix"].str.len()
distribution_zip=zip_length.value_counts()
correct_length=distribution_zip.get(5,0)
print(f"There are {correct_length} with 5 figure length")

if correct_length == len(df_geolocation):
    print("Every zip-code has the correct length.")
else:
    incorrect_rows = len(df_geolocation) - correct_length
    print(f"There are {incorrect_rows} errors in the column.")

There are 19015 with 5 figure length
Every zip-code has the correct length.


## Setting a index as surrogate key 

Max length of the strings int he column will be determined and an index will be set. Unusual names will be dropped, since there are no cities in brazil with more than 32 characters.

In [16]:

city_counts = df_geolocation['geolocation_city'].value_counts()


valid_cities = city_counts[city_counts >= 2].index


df_geolocation = df_geolocation[df_geolocation['geolocation_city'].isin(valid_cities)]
largest_city=df_geolocation['geolocation_city'].str.len().max()
print(f"max characters of the city column: {largest_city}")

df_geolocation.info()


max characters of the city column: 38
<class 'pandas.core.frame.DataFrame'>
Index: 13961 entries, 0 to 999743
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   geolocation_zip_code_prefix  13961 non-null  string 
 1   geolocation_lat              13961 non-null  float64
 2   geolocation_lng              13961 non-null  float64
 3   geolocation_city             13961 non-null  string 
 4   geolocation_state            13961 non-null  string 
dtypes: float64(2), string(3)
memory usage: 654.4 KB


In [46]:
df_geolocation=df_geolocation.reset_index(drop=True)#
df_geolocation['id_geolocation']=df_geolocation.index+1
df_geolocation=df_geolocation.set_index('id_geolocation')
display(df_geolocation)
df_geolocation.to_csv(r"C:\Users\User\Desktop\DSI\Abschlussprojekt\Quelldaten\silver_geolocation.csv",index=True,encoding='utf-8')
df_geolocation.info()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
id_geolocation,,,,,
1,01037,-23.545621,-46.639292,sao paulo,sp
2,01046,-23.546081,-46.644820,sao paulo,sp
3,01041,-23.544392,-46.639499,sao paulo,sp
4,01035,-23.541578,-46.641607,sao paulo,sp
5,01012,-23.547762,-46.635361,sao paulo,sp
...,...,...,...,...,...
13957,99702,-27.625951,-52.246655,erechim,rs
13958,99838,-27.763198,-52.001611,centenario,rs
13959,99830,-27.586163,-52.091061,gaurama,rs


<class 'pandas.core.frame.DataFrame'>
Index: 13961 entries, 1 to 13961
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   geolocation_zip_code_prefix  13961 non-null  string 
 1   geolocation_lat              13961 non-null  float64
 2   geolocation_lng              13961 non-null  float64
 3   geolocation_city             13961 non-null  string 
 4   geolocation_state            13961 non-null  string 
dtypes: float64(2), string(3)
memory usage: 654.4 KB


## Loading and inspecting the data

In [18]:
df2=pd.read_csv(r"C:\Users\User\Desktop\DSI\Abschlussprojekt\Quelldaten\bronze_product_category_name_translation.csv")
df_product_name_translation=df2.copy()
df_product_name_translation.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


# Transform the data

Transform datatype object into stringtype.

In [19]:
df_product_name_translation['product_category_name'].apply(type).value_counts()
df_product_name_translation['product_category_name_english'].apply(type).value_counts()

spalten_liste= ['product_category_name','product_category_name_english']
df_product_name_translation=df_product_name_translation[spalten_liste].astype("string")
df_product_name_translation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     string
 1   product_category_name_english  71 non-null     string
dtypes: string(2)
memory usage: 1.2 KB


# Data Cleaning

Removing whitespaces and transform it into snake case strings. Checking the strings for special charakters and removing duplicates.

In [20]:
df_product_name_translation['product_category_name'] = df_product_name_translation['product_category_name'].str.strip().str.lower()
df_product_name_translation['product_category_name_english'] = df_product_name_translation['product_category_name_english'].str.strip().str.lower()

string_elements="".join(df_product_name_translation['product_category_name'])
unique_strings=sorted(list(set(string_elements)))
print(unique_strings)

string_elements="".join(df_product_name_translation['product_category_name_english'])
unique_strings=sorted(list(set(string_elements)))
print(unique_strings)
df_product_name_translation.drop_duplicates()


['2', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
['2', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y']


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
...,...,...
66,flores,flowers
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes


## Index as surrogate key 

The index will be the surrogate key. The max. string length will be estimated to create a clean and fast database.

In [45]:
df_product_name_translation=df_product_name_translation.reset_index(drop=True)
df_product_name_translation['id_product_name_translation']=df_product_name_translation.index+1
df_product_name_translation=df_product_name_translation.set_index('id_product_name_translation')

max_length_portuguese=df_product_name_translation['product_category_name'].str.len().max()
max_length_english=df_product_name_translation['product_category_name'].str.len().max()

print(f"The max length of characters of the portugese column is {max_length_portuguese}.")
print(f"The max length of characters of the english column is {max_length_english}.")

df_product_name_translation.info()

display(df_product_name_translation)
df_product_name_translation.to_csv('silver_product_category_name_translation.csv',index=True,encoding='utf-8')
df_product_name_translation.to_csv(r"C:\Users\User\Desktop\DSI\Abschlussprojekt\Quelldaten\silver_product_name_translation.csv",index=False,encoding='utf-8')

The max length of characters of the portugese column is 46.
The max length of characters of the english column is 46.
<class 'pandas.core.frame.DataFrame'>
Index: 71 entries, 1 to 71
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     string
 1   product_category_name_english  71 non-null     string
dtypes: string(2)
memory usage: 1.7 KB


,product_category_name,product_category_name_english
id_product_name_translation,,
1,beleza_saude,health_beauty
2,informatica_acessorios,computers_accessories
3,automotivo,auto
4,cama_mesa_banho,bed_bath_table
5,moveis_decoracao,furniture_decor
...,...,...
67,flores,flowers
68,artes_e_artesanato,arts_and_craftmanship
69,fraldas_higiene,diapers_and_hygiene


# Inspecting the order items table

This table shows the order amount.

In [22]:
df3=pd.read_csv(r"C:\Users\User\Desktop\DSI\Abschlussprojekt\Quelldaten\bronze_olist_order_items_dataset.csv")
df_order_items=df3.copy()
df_order_items.info()
df_order_items


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


## Transforming the data

First the object will be inspected and see if it matches the index. Secondly, it is assessed if there are different datatypes in the object.

In [23]:
check_order=df_order_items['order_id'].apply(type).value_counts()
check_product=df_order_items['product_id'].apply(type).value_counts()
check_seller=df_order_items['seller_id'].apply(type).value_counts()
check_date=df_order_items['shipping_limit_date'].apply(type).value_counts()

print(check_date)
print(check_order)
print(check_product)
print(check_date)

shipping_limit_date
<class 'str'>    112650
Name: count, dtype: int64
order_id
<class 'str'>    112650
Name: count, dtype: int64
product_id
<class 'str'>    112650
Name: count, dtype: int64
shipping_limit_date
<class 'str'>    112650
Name: count, dtype: int64


There are no mixed types in the object, so the columns will be transferred into strings and the datetime64 in datetimeformat. There are no missing values sinced the count matches the rangeIndex.

In [24]:
df_order_items['shipping_limit_date'].apply(type).value_counts()
df_order_items['shipping_limit_date']=df_order_items['shipping_limit_date'].astype('datetime64[ns]')


df_order_items['seller_id'].apply(type).value_counts()
df_order_items['seller_id']=df_order_items['seller_id'].astype('string')


df_order_items['order_id'].apply(type).value_counts()
df_order_items['order_id']=df_order_items['order_id'].astype('string')


df_order_items['product_id'].apply(type).value_counts()
df_order_items['product_id']=df_order_items['product_id'].astype('string')
df_order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  string        
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  string        
 3   seller_id            112650 non-null  string        
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), string(3)
memory usage: 6.0 MB


## Data Validation

The colums will be checked in terms of unusal long string, special characters, unreasonable values or null values. There none null values detected

In [25]:
zero_values_order_value=df_order_items['order_item_id'].isna().sum()
print(f"There are {zero_values_order_value} null values in column order item id (amount)")
zero_values_order_price=df_order_items['price'].isna().sum()
print(f"There are {zero_values_order_price} null values in thecolumn price.")
zero_values_order_freight_value=df_order_items['freight_value'].isna().sum()
print(f"There are {zero_values_order_freight_value} null values in column freight value ")

There are 0 null values in column order item id (amount)
There are 0 null values in thecolumn price.
There are 0 null values in column freight value 


In [26]:
df_order_items['order_id'] = df_order_items['order_id'].str.strip().str.lower()

df_order_items['product_id'] = df_order_items['product_id'].str.strip().str.lower()
df_order_items['seller_id'] = df_order_items['seller_id'].str.strip().str.lower()

string_elements="".join(df_order_items['order_id'])
unique_strings=sorted(list(set(string_elements)))
print(unique_strings)

string_elements="".join(df_order_items['product_id'])
unique_strings=sorted(list(set(string_elements)))
print(unique_strings)

string_elements="".join(df_order_items['seller_id'])
unique_strings=sorted(list(set(string_elements)))
print(unique_strings)



['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f']
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f']
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f']


# Data Validation

In [27]:
min_qty = df_order_items['order_item_id'].min()
max_qty = df_order_items['order_item_id'].max()

print(f"Order quantity minimum: {min_qty} and maximum: {max_qty}")




Order quantity minimum: 1 and maximum: 21


Checking for wrong values. Since the divison operation itself can produce figures with more than 2 figures after the ., it will converted into strings to assure the correct format.

In [28]:
min_freight = df_order_items['freight_value'].min()
max_freight = df_order_items['freight_value'].max()

print(f"Shipping cost minimum: {min_freight} and maximum: {max_freight}")

wrong_floats_freight = ((df_order_items['price'] * 100) % 1 != 0).sum()

print(f"Prices with to many figures after the .: {wrong_floats_freight} rows")

freight_as_string = df_order_items['price'].round(2).astype(str)
freight_check = freight_as_string.str.split('.').str[1].str.len()

print(f" Max. number of figures after the .:{freight_check.max()}")


Shipping cost minimum: 0.0 and maximum: 409.68
Prices with to many figures after the .: 12765 rows
 Max. number of figures after the .:2


In [29]:
min_price = df_order_items['price'].min()
max_price = df_order_items['price'].max()

print(f"Order value  minimum: {min_price} and maximum: {max_price}")

wrong_floats = ((df_order_items['price'] * 100) % 1 != 0).sum()

print(f"Prices with to many figures after the .: {wrong_floats} rows")
df_order_items['price'] = df_order_items['price'].round(2)

float_check = ((df_order_items['price'] * 100) % 1 != 0).sum()

print(f"Prices with to many figures after the .: {float_check} rows")


price_as_string = df_order_items['price'].round(2).astype(str)
float_check = price_as_string.str.split('.').str[1].str.len()


print(f"Figures after the point : {float_check.max()}")

Order value  minimum: 0.85 and maximum: 6735.0
Prices with to many figures after the .: 12765 rows
Prices with to many figures after the .: 12765 rows
Figures after the point : 2


## Checking the key columns

All keys have length of 32 characters in 112650 rows. There are no unusual patterns.

In [30]:
order_id_len = df_order_items['order_id'].str.len().value_counts()
product_id_len = df_order_items['product_id'].str.len().value_counts()
seller_id_len = df_order_items['seller_id'].str.len().value_counts()


print(f"The len of the {order_id_len}")


print(f"The len of the {product_id_len}")


print(f"The len of the {seller_id_len}")



The len of the order_id
32    112650
Name: count, dtype: Int64
The len of the product_id
32    112650
Name: count, dtype: Int64
The len of the seller_id
32    112650
Name: count, dtype: Int64


## Setting an surrogate key

In [31]:
df_order_items=df_order_items.reset_index(drop=True)
df_order_items['id_olist_order_items']=df_order_items.index+1
df_order_items=df_order_items.set_index('id_olist_order_items')
df_order_items.drop_duplicates()


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
id_olist_order_items,,,,,,,
1,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
2,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
3,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
4,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
5,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112646,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112647,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112648,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95


## Checking the amount and the price with basic statistics

The statistics show,  that chiefly small orders are put. The standart deviation is with 0.7 low. There are a couple of very large orders. The price column reflects this. This can be senn looking at the median. Since the price is a larger figure, the other figures rise. The std is larger than the mean. There is at the max larger orders.

In [32]:
df_order_items.describe()

,order_item_id,shipping_limit_date,price,freight_value
count,112650.000000,112650,112650.000000,112650.000000
mean,1.197834,2018-01-07 15:36:52.192685312,120.653739,19.990320
min,1.000000,2016-09-19 00:15:34,0.850000,0.000000
25%,1.000000,2017-09-20 20:57:27.500000,39.900000,13.080000
50%,1.000000,2018-01-26 13:59:35,74.990000,16.260000
75%,1.000000,2018-05-10 14:34:00.750000128,134.900000,21.150000
max,21.000000,2020-04-09 22:35:08,6735.000000,409.680000
std,0.705124,NaN,183.633928,15.806405


## Reorder the columns the make the table clearly unstandable

In [44]:
reorder_columns= ['order_id','product_id','seller_id','shipping_limit_date','order_item_id','price','freight_value']
df_order_items=df_order_items[reorder_columns]
df_order_items
display(df_order_items)
df_order_items.to_csv(r"C:\Users\User\Desktop\DSI\Abschlussprojekt\Quelldaten\silver_order_items.csv",index=True,encoding='utf-8')
display(df_order_items)

,order_id,product_id,seller_id,shipping_limit_date,order_item_id,price,freight_value
id_olist_order_items,,,,,,,
1,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,1,58.90,13.29
2,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,1,239.90,19.93
3,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,1,199.00,17.87
4,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,1,12.99,12.79
5,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,1,199.90,18.14
...,...,...,...,...,...,...,...
112646,fffc94f6ce00a00581880bf54a75a037,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,1,299.99,43.41
112647,fffcd46ef2263f404302a634eb57f7eb,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,1,350.00,36.53
112648,fffce4705a9662cd70adb13d4a31832d,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,1,99.90,16.95


,order_id,product_id,seller_id,shipping_limit_date,order_item_id,price,freight_value
id_olist_order_items,,,,,,,
1,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,1,58.90,13.29
2,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,1,239.90,19.93
3,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,1,199.00,17.87
4,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,1,12.99,12.79
5,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,1,199.90,18.14
...,...,...,...,...,...,...,...
112646,fffc94f6ce00a00581880bf54a75a037,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,1,299.99,43.41
112647,fffcd46ef2263f404302a634eb57f7eb,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,1,350.00,36.53
112648,fffce4705a9662cd70adb13d4a31832d,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,1,99.90,16.95
